## Installing necessary libraries of python 

In [1]:
%pip install numpy
%pip install pandas
%pip install requests
%pip install beautifulsoup4
%pip install lxml
%pip install selenium webdriver-manager beautifulsoup4

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import re


In [3]:
headers = {
"User-Agent": "Mozilla/5.0"
}
url="https://www.magicbricks.com/property-for-rent-in-ahmedabad-pppfr"
webpage = requests.get(url, headers=headers).text
soup=BeautifulSoup(webpage,"lxml")
properties = soup.find_all("div", class_="mb-srp__card")
print(len(properties))

30


In [4]:
links = []

for tag in soup.select("a[href*='propertyDetails']"):
    link = tag.get("href")

    if not link.startswith("http"):
        link = "https://www.magicbricks.com" + link

    links.append(link)

In [5]:
len(links)

30

In [6]:
link=links[0]
link

'https://www.magicbricks.com/propertyDetails/3-BHK-1900-Sq-ft-Multistorey-Apartment-FOR-Rent-Makarba-in-Ahmedabad&id=4d423833373833393131'

In [7]:
url1=link
response1=requests.get(url1,headers=headers)
soup1=BeautifulSoup(response1.text, "lxml")
print(len(soup1))

2


In [8]:
all_html = soup1.find_all("li", class_="mb-ldp__dtls__nav__list--item")
for html in all_html:
    print(html.text.strip())


Overview
More Details
About Project


In [9]:
price=(soup1.find("div",class_="mb-ldp__dtls__price").text.strip())
print(price)

₹52,000


In [10]:
Title=(soup1.find("span",class_="mb-ldp__dtls__title--text1--text pad-r-4").text.strip())
print(Title)


3 BHK Flat 1900 Sq-ft For Rent in Orchid Woods,


In [11]:
location=(soup1.find("a",class_="mb-ldp__dtls__title--link").text.strip())
print(location)

Makarba, Ahmedabad


In [12]:
body=soup1.find('div',class_="mb-ldp__dtls__body")
first=body.find('ul',class_="mb-ldp__dtls__body__summary")
second=body.find('ul',class_="mb-ldp__dtls__body__list")
#print(first)

In [13]:
row1={}

In [14]:
items = first.find_all("li")

for item in items:

    value = item.find("span", class_="mb-ldp__dtls__body__summary--highlight")

    if value:

        val = value.text.strip()
        key = item.text.replace(val, "").strip()

        # only store if both key and value exist
        if key and val:
            row1[key] = val

row1

{'Beds': '3', 'Baths': '3', 'Balcony': '1'}

In [15]:
row2={}

In [16]:
if second:

    items = second.find_all("li")

    for item in items:

        label = item.find("div", class_="mb-ldp__dtls__body__list--label")
        value = item.find("div", class_="mb-ldp__dtls__body__list--value")

        if label and value:

            key = label.text.strip()
            val = value.text.strip()

            # avoid empty columns
            if key and val:
                row2[key] = val
row2

{'Super Built-up Area': '1900sqftsqftsqyrdsqmacrebighahectaremarlakanalbiswa1biswa2groundaankadamroodchatakkottahmarlacentperchgunthaarekathagajkillakuncham₹27/sqft',
 'Developer': 'Goyal & Co.',
 'Project': 'Orchid Woods',
 'Floor': '4(Out of 11 Floors)',
 'Facing': 'South - East',
 'Furnished Status': 'Furnished',
 'Age Of Construction': '10 to 15 years'}

In [17]:
def Overview(soup1, row):

    # title
    Title = soup1.find("span", class_="mb-ldp__dtls__title--text1--text pad-r-4")
    row["Title"] = Title.text.strip() if Title else None

    # price
    price = soup1.find("div", class_="mb-ldp__dtls__price")
    row["Rental_Price"] = price.text.strip() if price else None

    # location
    location = soup1.find("a", class_="mb-ldp__dtls__title--link")
    row["Location"] = location.text.strip() if location else None

    # main body
    body = soup1.find("div", class_="mb-ldp__dtls__body")

    if not body:
        return row

    # -------- summary section --------
    first = body.find("ul", class_="mb-ldp__dtls__body__summary")

    if first:

        items = first.find_all("li")

        for item in items:

            value = item.find("span", class_="mb-ldp__dtls__body__summary--highlight")

            if value:

                val = value.text.strip()
                key = item.text.replace(val, "").strip()

                if key and val:
                    row[key] = val

    # -------- more details section --------
    second = body.find("ul", class_="mb-ldp__dtls__body__list")

    if second:

        items = second.find_all("li")

        for item in items:

            label = item.find("div", class_="mb-ldp__dtls__body__list--label")
            value = item.find("div", class_="mb-ldp__dtls__body__list--value")

            if label and value:

                key = label.text.strip()
                val = value.text.strip()

                if key and val:
                    row[key] = val

    return row

In [18]:
body2=soup1.find_all('li',class_="mb-ldp__more-dtl__list--item")

In [19]:
row3={}

In [20]:
items = soup1.find_all("li", class_="mb-ldp__more-dtl__list--item")
for item in items:

    label = item.find("div", class_="mb-ldp__more-dtl__list--label")
    value = item.find("div", class_="mb-ldp__more-dtl__list--value")

    if label and value:

        key = label.get_text(strip=True)
        val = value.get_text(strip=True)

        row3[key] = val
row3

{'Rental Value': '₹52,000|₹12,600Quarterly Maintenance',
 'Security Deposit': '₹1.0 Lac',
 'Address': 'Makarba, Ahmedabad - West, Gujarat',
 'Furnishing': 'Furnished',
 'Overlooking': 'Garden/Park, Main Road',
 'Age of Construction': '10 to 15 years'}

In [21]:
def More_Details(soup1, row):

    items = soup1.find_all("li", class_="mb-ldp__more-dtl__list--item")

    for item in items:

        label = item.find("div", class_="mb-ldp__more-dtl__list--label")
        value = item.find("div", class_="mb-ldp__more-dtl__list--value")

        if label and value:

            key = label.get_text(strip=True)
            val = value.get_text(strip=True)

            if key and val:
                row[key] = val

    return row

In [22]:
body4=soup1.find('ul',class_="mb-ldp__about-proj__list")
#body4.find_all('li',class_="mb-ldp__about-proj__list--item")

In [23]:
body4=soup1.find('ul',class_="mb-ldp__about-proj__list")
#body4.find_all('li',class_="mb-ldp__about-proj__list--item")

In [24]:
row4={}

In [25]:
body4 = soup1.find('ul',class_="mb-ldp__amenities__list")
amenities=""    
if body4:
    
    items = body4.find_all('li', class_="mb-ldp__amenities__list--item")
    
    for item in items:
        amenities=item.text +','+ amenities
row4['amenities']=amenities

In [26]:
body5 = soup1.find_all('li', class_="mb-ldp__about-proj__list--item")

for item in body5:

    label = item.find("div", class_="mb-ldp__about-proj__list--label")
    value = item.find("div", class_="mb-ldp__about-proj__list--value")

    if label and value:

        key = label.get_text(strip=True)
        val = value.get_text(" ", strip=True)

        row4[key] = val

In [27]:
row4

{'amenities': '',
 'Price': '₹ 1.30 Cr - ₹ 1.62 Cr',
 'Configuration': '3, 4 BHK Flat',
 'Tower & Unit': '2 Towers , 96 Units'}

In [28]:
def About_Project(soup1, row):

    # -------- Amenities --------
    body4 = soup1.find('ul', class_="mb-ldp__amenities__list")

    amenities = ""

    if body4:

        items = body4.find_all('li', class_="mb-ldp__amenities__list--item")

        for item in items:
            amenities = item.get_text(strip=True) + "," + amenities

        row["Amenities"] = amenities.strip(",")

    # -------- Project Details --------
    body5 = soup1.find_all('li', class_="mb-ldp__about-proj__list--item")

    for item in body5:

        label = item.find("div", class_="mb-ldp__about-proj__list--label")
        value = item.find("div", class_="mb-ldp__about-proj__list--value")

        if label and value:

            key = label.get_text(strip=True)
            val = value.get_text(" ", strip=True)

            if key and val:
                row[key] = val

    return row

In [ ]:
headers = {
    "User-Agent": "Mozilla/5.0"
}

cities = ["ahmedabad"] #["mumbai","kolkata","bangalore","new-delhi","hyderabad","chennai","pune","ahmedabad"]

for city in cities:

    base_url = "https://www.magicbricks.com/property-for-rent-in-{}-pppfr/page-{}"

    data = []
    pages = 110

    for page in range(1, pages + 1):

        print(f"Scraping Page {page} of {city}")

        url = base_url.format(city, page)

        try:

            response = requests.get(url, headers=headers, timeout=10)

            if response.status_code != 200:
                print("Page blocked or not found")
                continue

            soup = BeautifulSoup(response.text, "lxml")

            # extract property links
            links = set()

            for tag in soup.select("a[href*='propertyDetails']"):

                link = tag.get("href")

                if link:
                    if not link.startswith("http"):
                        link = "https://www.magicbricks.com" + link

                    links.add(link)

            print("Total properties found:", len(links))

            # visit each property page
            for link in links:

                row = {}

                try:

                    response1 = requests.get(link, headers=headers, timeout=10)

                    if response1.status_code != 200:
                        continue

                    soup1 = BeautifulSoup(response1.text, "lxml")

                    sections = soup1.find_all(
                        "li",
                        class_="mb-ldp__dtls__nav__list--item"
                    )

                    for section in sections:

                        section_name = section.text.strip()

                        if section_name == "Overview":
                            row = Overview(soup1, row)

                        if section_name == "More Details":
                            row = More_Details(soup1, row)

                        if section_name == "About Project":
                            row = About_Project(soup1, row)

                    
                    row["link"] = link

                    data.append(row)

                    time.sleep(0)

                except Exception as e:
                    print("Property error:", e)

        except Exception as e:
            print(f"Error on page {page} of {city}: {e}")
            break

    df = pd.DataFrame(data)

    df.to_csv(f"Data\{city}_new_rent_data.csv", index=False)

    print("CSV created successfully")


Scraping Page 1 of ahmedabad
Total properties found: 30
